## Etapa 4 — Modelagem Preditiva e Produto Analítico para Antecipação do NPS

Nesta etapa, a análise evolui de diagnóstico para **ação preventiva**.

A proposta é utilizar os sinais operacionais identificados na EDA para antecipar quais clientes têm maior risco de se tornarem detratores antes da aplicação da pesquisa de NPS.

Para uma liderança de operações, a pergunta deixa de ser apenas:

**O que aconteceu com o NPS?**

E passa a ser:

**Quais clientes devo priorizar hoje para evitar uma experiência negativa?**

### Objetivo da modelagem

O objetivo da modelagem não é apenas construir um algoritmo, mas criar uma lógica prática de priorização operacional.

A partir de variáveis como atraso logístico, contatos com atendimento, tempo de resolução e reclamações, o modelo deve gerar:

- uma probabilidade de risco de detrator;
- uma segmentação de clientes por nível de risco;
- uma regra de ação recomendada;
- uma fila de priorização para a operação.

Essa abordagem conecta Ciência de Dados com decisão real no dia a dia do negócio.

### Estratégia adotada: classificação

Para este problema, a abordagem recomendada é um **modelo de classificação**.

Em vez de prever exatamente a nota de NPS em uma escala de 0 a 10, o modelo busca identificar se um cliente tem risco de se tornar detrator.

Essa estratégia é mais útil para a operação, pois transforma o problema em uma decisão prática:

**Este cliente precisa de ação preventiva?**

1 - Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.max_columns", None)

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# sns.set_theme(style="whitegrid", palette="pastel")
sns.set_theme(style="ticks")

DATA_PATH = "../data/raw/desafio_nps_fase_1.csv"


In [2]:
df = pd.read_csv(DATA_PATH)

1.Variável alvo

In [3]:
# Criação da variável alvo para classificação
# 1 = cliente detrator
# 0 = cliente neutro ou promotor

df["is_detractor"] = (df["nps_score"] <= 6).astype(int)

# Distribuição da variável alvo em percentual
distribuicao_target = (df["is_detractor"].value_counts(normalize=True) * 100).round(1)

distribuicao_target

is_detractor
1    74.0
0    26.0
Name: proportion, dtype: float64

In [4]:
df["is_detractor"].unique()

array([0, 1])

### Definição da variável alvo

A variável original de satisfação é o NPS.

Para permitir aplicação operacional do modelo, foi criada, a partir do `nps_score`, a variável `is_detractor`, definida como:

- `1`: cliente detrator, com NPS de 0 a 6;
- `0`: cliente neutro ou promotor, com NPS acima de 6.

Essa definição permite transformar o problema em uma decisão prática: identificar clientes com maior risco de insatisfação.

2. Variável de entrada

Seleciona as variáveis que serão usadas para prever o risco de detrator.

In [5]:
# Seleção das variáveis de entrada
# Foram escolhidas variáveis operacionais disponíveis antes da coleta do NPS.

features = [
    "customer_age",
    "customer_region",
    "customer_tenure_months",
    "order_value",
    "items_quantity",
    "discount_value",
    "payment_installments",
    "delivery_time_days",
    "delivery_delay_days",
    "freight_value",
    "delivery_attempts",
    "customer_service_contacts",
    "resolution_time_days",
    "complaints_count"
]

X = df[features]
y = df["is_detractor"]

# Visualizando as primeiras linhas das variáveis de entrada
X.head()

,customer_age,customer_region,customer_tenure_months,order_value,items_quantity,discount_value,payment_installments,delivery_time_days,delivery_delay_days,freight_value,delivery_attempts,customer_service_contacts,resolution_time_days,complaints_count
0,63,Nordeste,14,139.73,4,39.35,4,2,2,55.53,3,0,4,3
1,20,Sul,1,458.95,2,9.51,10,6,4,28.23,3,0,10,3
2,46,Nordeste,111,507.06,5,42.82,6,6,1,40.99,1,4,5,7
3,52,Centro-Oeste,117,302.19,2,19.58,9,5,2,35.24,3,1,11,4
4,56,Norte,50,253.06,1,29.37,11,13,1,39.32,1,1,0,3


### Seleção das variáveis de entrada

Foram utilizadas variáveis relacionadas ao perfil do cliente, pedido, logística e atendimento.

Variáveis como `customer_id`, `order_id` e `nps_score` não foram usadas como entrada, pois não ajudam a prever o comportamento operacional futuro ou poderiam gerar vazamento de informação.

A variável `repeat_purchase_30d` também não foi utilizada, pois representa um comportamento posterior à experiência e poderia tornar a previsão artificialmente otimista.

3. Treino e Teste

Divide os dados em:

- treino: usado para o modelo aprender

- teste: usado para avaliar se o modelo funciona em dados novos

In [6]:
# Separação entre treino e teste
# Treino: usado para o modelo aprender os padrões históricos
# Teste: usado para avaliar o desempenho em dados não vistos

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Tamanho do treino:", X_train.shape)
print("Tamanho do teste:", X_test.shape)

print("\nDistribuição da target no treino (%):")
print((y_train.value_counts(normalize=True) * 100).round(1))

print("\nDistribuição da target no teste (%):")
print((y_test.value_counts(normalize=True) * 100).round(1))

Tamanho do treino: (2000, 14)
Tamanho do teste: (500, 14)

Distribuição da target no treino (%):
is_detractor
1    74.1
0    26.0
Name: proportion, dtype: float64

Distribuição da target no teste (%):
is_detractor
1    74.0
0    26.0
Name: proportion, dtype: float64


### Lógica da separação dos dados

A base foi dividida em treino e teste para validar se o modelo consegue generalizar para novos clientes.

O parâmetro `stratify=y` mantém a proporção de detratores e não detratores nos dois conjuntos, evitando distorções na avaliação.

4. Preparação da variáveis

Prepara os dados para os modelos:

- variáveis numéricas são padronizadas

- variável categórica customer_region é convertida em colunas numéricas

In [7]:
# Separação das variáveis numéricas e categóricas

numeric_features = [
    "customer_age",
    "customer_tenure_months",
    "order_value",
    "items_quantity",
    "discount_value",
    "payment_installments",
    "delivery_time_days",
    "delivery_delay_days",
    "freight_value",
    "delivery_attempts",
    "customer_service_contacts",
    "resolution_time_days",
    "complaints_count"
]

categorical_features = ["customer_region"]

# Pipeline de preparação:
# - variáveis numéricas são padronizadas
# - variável categórica é convertida em colunas numéricas

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

### Preparação das variáveis

A preparação dos dados garante que todos os modelos recebam as variáveis em formato adequado.

As variáveis numéricas foram padronizadas e a variável categórica `customer_region` foi transformada em representação numérica por meio de One Hot Encoding.

Essa preparação foi organizada em uma pipeline para tornar o processo mais reproduzível e evitar inconsistências entre treino e teste.

5. Comparação do modelos

Treina três modelos diferentes e compara os resultados:

- Regressão Logística

- Random Forest

- Gradient Boosting

A métrica principal será recall, porque queremos encontrar o maior número possível de clientes detratores.

In [8]:
# Comparação de modelos
# A métrica principal será o Recall, pois o objetivo é encontrar o maior número possível de detratores.

models = {
    "Regressão Logística": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(random_state=42, class_weight="balanced"),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

results = []

for model_name, classifier in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "Modelo": model_name,
        "Acurácia": accuracy_score(y_test, y_pred),
        "Precisão": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred),
        "F1-score": f1_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_proba)
    })

df_results = (
    pd.DataFrame(results)
    .sort_values(by=["Recall", "F1-score", "AUC"], ascending=False)
    .round(3)
)

df_results

,Modelo,Acurácia,Precisão,Recall,F1-score,AUC
1,Random Forest,0.832,0.852,0.935,0.892,0.870
2,Gradient Boosting,0.832,0.863,0.919,0.890,0.873
0,Regressão Logística,0.788,0.915,0.786,0.846,0.877


### Comparação dos modelos

Foram comparados três modelos de classificação:

- **Regressão Logística**: modelo mais simples e interpretável;
- **Random Forest**: modelo baseado em múltiplas árvores de decisão, capaz de capturar relações não lineares;
- **Gradient Boosting**: modelo mais robusto, que aprende de forma sequencial com os erros anteriores.

Como o objetivo do negócio é identificar clientes com risco de se tornarem detratores, a métrica mais importante é o **Recall**.

Na base analisada, o modelo com melhor desempenho em Recall foi **Random Forest**, com Recall de aproximadamente **0.935** e AUC de **0.870**.

Do ponto de vista operacional, isso significa que esse modelo apresentou maior capacidade de identificar clientes em risco antes da pesquisa de NPS.

### Como interpretar as métricas

Para este problema, as métricas devem ser interpretadas com foco no uso operacional:

- **Recall**: mostra quantos detratores reais o modelo conseguiu identificar;
- **Precisão**: mostra quantos clientes sinalizados como risco realmente eram detratores;
- **F1-score**: equilibra precisão e recall;
- **AUC**: mede a capacidade geral do modelo de diferenciar clientes em risco e sem risco.

Neste caso, deixar um detrator passar sem ação preventiva é mais crítico do que acionar alguns clientes a mais. Por isso, o Recall tem maior peso na escolha do modelo.

In [9]:
# Seleção automática do melhor modelo com base no Recall

best_model_name = df_results.iloc[0]["Modelo"]
best_classifier = models[best_model_name]

best_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", best_classifier)
])

best_model.fit(X_train, y_train)

print("Modelo selecionado:", best_model_name)


Modelo selecionado: Random Forest


### Escolha do modelo

A escolha do modelo foi orientada pelo equilíbrio entre desempenho e aplicação prática.

O modelo selecionado foi aquele com maior capacidade de identificar clientes detratores, priorizando o Recall.

Para o negócio, isso significa aumentar a chance de encontrar clientes críticos antes da pesquisa de satisfação, permitindo que a operação atue preventivamente.

6.1 Avaliação detalhada do modelo escolhido - **RANDOM FOREST**

Mostra a performance detalhada do modelo final.

In [10]:
# Avaliação detalhada do modelo selecionado

y_pred_best = best_model.predict(X_test)
y_proba_best = best_model.predict_proba(X_test)[:, 1]

print("Relatório de classificação:")
print(classification_report(y_test, y_pred_best))

print("Matriz de confusão:")
print(confusion_matrix(y_test, y_pred_best))

print("AUC:", round(roc_auc_score(y_test, y_proba_best), 3))

Relatório de classificação:
              precision    recall  f1-score   support

           0       0.74      0.54      0.62       130
           1       0.85      0.94      0.89       370

    accuracy                           0.83       500
   macro avg       0.80      0.74      0.76       500
weighted avg       0.82      0.83      0.82       500

Matriz de confusão:
[[ 70  60]
 [ 24 346]]
AUC: 0.87


### Interpretação executiva da avaliação

A matriz de confusão mostra quatro situações:

- **Verdadeiros positivos**: detratores identificados corretamente;
- **Falsos negativos**: detratores que o modelo não identificou;
- **Falsos positivos**: clientes sinalizados como risco, mas que não eram detratores;
- **Verdadeiros negativos**: clientes sem risco identificados corretamente.

Para a operação, o ponto mais sensível são os falsos negativos, pois representam clientes insatisfeitos que não seriam priorizados.

Por isso, a modelagem deve ser avaliada não apenas pela acurácia, mas pela capacidade de reduzir o risco de deixar detratores sem ação.

7. Criação do score de risco

Cria uma probabilidade de risco para cada cliente da base de teste.

In [11]:
# Criação do score de risco
# O risk_score representa a probabilidade estimada de o cliente se tornar detrator.

df_produto = X_test.copy()

df_produto["real_is_detractor"] = y_test.values
df_produto["risk_score"] = np.round(y_proba_best, 3)

df_produto[["risk_score", "real_is_detractor"]].head()

,risk_score,real_is_detractor
1005,0.46,1
2151,0.67,1
76,0.86,1
441,0.89,1
1875,0.68,0


### Score de risco

O `risk_score` transforma a previsão do modelo em uma probabilidade de risco.

Quanto maior o score, maior a chance de o cliente se tornar detrator.

Esse score é o ponto de partida para transformar o modelo em um produto analítico para a operação.

7.1 - Segmentação de clientes

Transforma a probabilidade do modelo em segmentos fáceis de usar pela operação.

In [12]:
# Segmentação dos clientes por nível de risco
# Os limites podem ser ajustados futuramente conforme a capacidade operacional da empresa.

def segmentar_cliente(score):
    if score >= 0.75:
        return "Crítico"
    elif score >= 0.50:
        return "Atenção"
    elif score >= 0.30:
        return "Monitoramento"
    else:
        return "Regular"

df_produto["segmento_risco"] = df_produto["risk_score"].apply(segmentar_cliente)

df_produto["segmento_risco"].value_counts()

segmento_risco
Crítico          316
Atenção           95
Monitoramento     56
Regular           33
Name: count, dtype: int64

### Segmentação de clientes por risco

A probabilidade gerada pelo modelo foi convertida em quatro segmentos operacionais:

- **Crítico**: clientes com alta probabilidade de se tornarem detratores;
- **Atenção**: clientes com sinais relevantes de fricção;
- **Monitoramento**: clientes com risco moderado;
- **Regular**: clientes sem sinais críticos relevantes.

Essa segmentação facilita o uso do modelo por áreas não técnicas, pois transforma uma probabilidade em uma regra clara de priorização.

8. Regra de Ação Operacional

A regra de ação traduz o resultado do modelo em uma recomendação prática.

Em vez de entregar apenas uma previsão estatística, o modelo passa a orientar o que a operação deve fazer com cada grupo de clientes.

| Segmento | Critério | Interpretação | Ação recomendada |
|---|---|---|---|
| Crítico | risco ≥ 75% | alta chance de detrator | atendimento imediato + comunicação proativa |
| Atenção | risco entre 50% e 74% | sinais relevantes de fricção | monitorar pedido + contato preventivo |
| Monitoramento | risco entre 30% e 49% | risco moderado | acompanhar indicadores operacionais |
| Regular | risco < 30% | baixo risco | fluxo padrão |

In [13]:
# Regra de ação recomendada por segmento
# Esta etapa transforma a previsão do modelo em recomendação prática para a operação.

def recomendar_acao(row):
    if row["segmento_risco"] == "Crítico":
        return "Atendimento imediato + comunicação proativa"
    elif row["segmento_risco"] == "Atenção":
        return "Monitorar pedido + contato preventivo"
    elif row["segmento_risco"] == "Monitoramento":
        return "Acompanhar indicadores operacionais"
    else:
        return "Fluxo padrão"

df_produto["acao_recomendada"] = df_produto.apply(recomendar_acao, axis=1)

df_produto[[
    "risk_score",
    "segmento_risco",
    "acao_recomendada"
]].head()

,risk_score,segmento_risco,acao_recomendada
1005,0.46,Monitoramento,Acompanhar indicadores operacionais
2151,0.67,Atenção,Monitorar pedido + contato preventivo
76,0.86,Crítico,Atendimento imediato + comunicação proativa
441,0.89,Crítico,Atendimento imediato + comunicação proativa
1875,0.68,Atenção,Monitorar pedido + contato preventivo


8.1 - Produto analítico: fila de priorização operacional

A tabela anterior representa o principal produto da modelagem.

Ela permite que a empresa saiba, no dia a dia:

- quem deve ser priorizado;
- qual cliente apresenta maior risco;
- quais sinais operacionais justificam o risco;
- qual ação deve ser tomada.

Essa fila pode ser usada diariamente pelas áreas de Operações, Logística, Atendimento ou Customer Experience para agir antes da pesquisa de NPS.

In [14]:
# Fila de priorização operacional
# Esta tabela simula uma lista diária para a equipe de Operações ou Customer Experience.

fila_priorizacao = df_produto.sort_values(
    by="risk_score",
    ascending=False
)

tabela_fila = fila_priorizacao[[
    "delivery_delay_days",
    "customer_service_contacts",
    "resolution_time_days",
    "complaints_count",
    "risk_score",
    "segmento_risco",
    "acao_recomendada"
]].head(10).copy()

tabela_fila

,delivery_delay_days,customer_service_contacts,resolution_time_days,complaints_count,risk_score,segmento_risco,acao_recomendada
1879,3,2,2,5,1.00,Crítico,Atendimento imediato + comunicação proativa
139,4,3,9,6,1.00,Crítico,Atendimento imediato + comunicação proativa
91,3,2,3,5,1.00,Crítico,Atendimento imediato + comunicação proativa
2395,3,3,2,5,1.00,Crítico,Atendimento imediato + comunicação proativa
2389,3,3,6,6,1.00,Crítico,Atendimento imediato + comunicação proativa
660,4,4,2,7,1.00,Crítico,Atendimento imediato + comunicação proativa
2309,5,4,5,6,1.00,Crítico,Atendimento imediato + comunicação proativa
1915,4,3,11,6,1.00,Crítico,Atendimento imediato + comunicação proativa
2215,5,2,8,6,1.00,Crítico,Atendimento imediato + comunicação proativa
877,5,2,3,5,0.99,Crítico,Atendimento imediato + comunicação proativa


8.2 - Resumo dos segmentos

Agrupa os clientes por segmento e mostra indicadores médios.

In [15]:
# Resumo executivo dos segmentos de risco
# Esta visão ajuda a liderança a entender o perfil operacional de cada grupo.

resumo_segmentos = df_produto.groupby("segmento_risco").agg(
    quantidade_clientes=("risk_score", "count"),
    risco_medio=("risk_score", "mean"),
    atraso_medio=("delivery_delay_days", "mean"),
    contatos_medios=("customer_service_contacts", "mean"),
    tempo_resolucao_medio=("resolution_time_days", "mean"),
    reclamacoes_medias=("complaints_count", "mean")
).reset_index()

resumo_segmentos = resumo_segmentos.sort_values(by="risco_medio", ascending=False).round(2)

resumo_segmentos

,segmento_risco,quantidade_clientes,risco_medio,atraso_medio,contatos_medios,tempo_resolucao_medio,reclamacoes_medias
1,Crítico,316,0.90,2.76,1.91,5.85,4.97
0,Atenção,95,0.63,1.48,1.04,4.97,3.38
2,Monitoramento,56,0.40,1.02,0.82,4.77,2.36
3,Regular,33,0.23,0.45,0.61,3.88,1.30


### Como usar o resumo dos segmentos

O resumo por segmento ajuda a liderança a entender se os grupos de maior risco realmente concentram mais fricções operacionais.

Na prática, espera-se que os grupos **Crítico** e **Atenção** apresentem maior risco médio e maior concentração de fatores como atraso, contatos, tempo de resolução e reclamações.

Essa visão permite dimensionar a operação e priorizar esforços onde o risco é maior.

### Conexão com o uso real

Na prática, o modelo poderia ser integrado a uma rotina diária da operação.

Exemplo de uso:

1. atualizar os dados operacionais de pedidos, logística e atendimento;
2. calcular o risco de detrator para cada cliente;
3. classificar clientes por segmento de risco;
4. gerar uma fila de priorização;
5. aplicar ações preventivas conforme a regra definida;
6. acompanhar se houve redução de detratores ao longo do tempo.

Dessa forma, a empresa deixa de atuar apenas após a insatisfação e passa a agir antes que o problema se transforme em uma avaliação negativa.

### Conclusão executiva da modelagem

A comparação de modelos permitiu avaliar diferentes abordagens para antecipar o risco de insatisfação do cliente.

A escolha do modelo foi orientada pelo Recall, pois o objetivo do negócio é identificar o maior número possível de clientes com risco de se tornarem detratores.

A partir do modelo selecionado, foi criado um score de risco, uma segmentação de clientes, uma regra de ação e uma fila de priorização operacional.

Com isso, a modelagem deixa de ser apenas uma etapa técnica e passa a funcionar como um **produto analítico de apoio à decisão**, capaz de orientar ações preventivas no dia a dia da operação.

A principal contribuição da modelagem não é apenas prever o NPS, mas transformar sinais operacionais em uma rotina prática de prevenção de detratores.